# Clean  BARRA2  data


Author: Aminath Shausan
 

#### **Load libraries**

In [11]:
path = '/Users/aminath/csiroDocuments/GitHub/climate_project/'
import warnings
warnings.filterwarnings("ignore")


In [ ]:
#import libraries
from datetime import datetime
import pandas as pd
import numpy as np
import xarray as xr # t
import geopandas as gpd #r
import regionmask #to determine which geographic region each grid point belongs to 

import cartopy.crs as ccrs # for plotting world map
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.lines import Line2D ## for legends 

import scipy
import scipy.linalg as linalg #for linear regression

import json  #for saving results
import pickle
from matplotlib import pyplot as plt

In [13]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors     

# from IPython.display import set_matplotlib_formats
# set_matplotlib_formats('png')
# plt.rcParams['savefig.dpi'] = 75
plt.rcParams['figure.autolayout'] = False
plt.rcParams['figure.figsize'] = (21, 30)  # A4
plt.rcParams['lines.linewidth'] = 2.0
plt.rcParams['lines.markersize'] = 8
#plt.rcParams['text.usetex'] = True
plt.rcParams['text.usetex'] = False
plt.rcParams['font.size'] = 16
plt.rcParams['legend.fontsize'] = 16
#plt.rcParams['font.family'] = "serif"
plt.rcParams['font.family'] = "sans"
matplotlib.rcParams['xtick.major.pad']=12
matplotlib.rcParams['ytick.major.pad']=12

## Some function

In [14]:
def aggregate_climate_data_per_jurisdiction(ds_climate, ds_pop_interp, shape_file, mask_reanalysis):
    state_short_name = dict()
    state_short_name['New South Wales'] = 'NSW'
    state_short_name['Victoria'] = 'VIC'
    state_short_name['Queensland'] = 'QLD'
    state_short_name['South Australia'] = 'SA'
    state_short_name['Western Australia'] = 'WA'
    state_short_name['Australian Capital Territory'] = 'ACT'
    state_short_name['Northern Territory'] = 'NT'
    state_short_name['Tasmania'] = 'TAS'  
    state_short_name['Australia'] = 'AUS'  

    list_of_ds_climate_avg = list()
    list_of_ds_climate_pop_weighted_avg = list()

    for region_name in state_short_name.keys():
        print('Processing {0}'.format(region_name))
        if region_name == 'Australia':
            ds_climate_region     = ds_climate.where(mask_reanalysis < 8)
            ds_pop_interp_region  = ds_pop_interp.popn.where(mask_reanalysis < 8)
        else:
            ds_climate_region     = ds_climate.where(mask_reanalysis == shape_file[shape_file.STE_NAME21==region_name].index[0])
            ds_pop_interp_region  = ds_pop_interp.popn.where(mask_reanalysis == shape_file[shape_file.STE_NAME21==region_name].index[0])

        ds_climate_region_avg = ds_climate_region.mean('lat').mean('lon')
        ds_climate_region_avg = ds_climate_region_avg.assign_coords({"jurisdiction": state_short_name[region_name]})
        for var_name in ['tas', 'RH']: ## ['wet_bulb_temp','tas', 'tasmax', 'tasmin', 'RH', 'pr']
            ds_climate_region_avg[var_name] = ds_climate_region_avg[var_name].expand_dims(dim='jurisdiction',axis=1)
        list_of_ds_climate_avg.append(ds_climate_region_avg) ; del ds_climate_region_avg

        ds_climate_region_pop_weighted_avg = ( ds_climate_region*ds_pop_interp_region / ds_pop_interp_region.sum('lat').sum('lon') ).sum('lat').sum('lon')
        ds_climate_region_pop_weighted_avg = ds_climate_region_pop_weighted_avg.assign_coords({"jurisdiction": state_short_name[region_name]})
        for var_name in ['tas','RH']: ##['wet_bulb_temp','tas', 'tasmax', 'tasmin', 'RH', 'pr']
            ds_climate_region_pop_weighted_avg[var_name] = ds_climate_region_pop_weighted_avg[var_name].expand_dims(dim='jurisdiction',axis=1)
        list_of_ds_climate_pop_weighted_avg.append(ds_climate_region_pop_weighted_avg) ; del ds_climate_region_pop_weighted_avg
        
        del ds_climate_region, ds_pop_interp_region

    print('Merging')
    ds_climate_avg = xr.merge(list_of_ds_climate_avg) ; del list_of_ds_climate_avg
    ds_climate_pop_weighted_avg = xr.merge(list_of_ds_climate_pop_weighted_avg) ; del list_of_ds_climate_pop_weighted_avg
    
    return ds_climate_avg, ds_climate_pop_weighted_avg

In [ ]:
# def calculate_WBGT(p_ref, T_ref, rh_ref, print_output=True):     
#     """Calculate the Wet-Bulb Globe Temperature as in Newth & Gunasekera (2018, Atmosphere)  
#     Inputs:
#         p_ref   is the daily sea level pressure in mbar
#         T_ref   is the daily 2m temperature in degrees Kelvin
#         rh_ref  is the daily 2m relative humidity as a percentage 
        
#     Local variables:
#         T_a     is the daily 2m temperature in degrees Celcius
#         T_L     is the daily 2m lifting condensation temperature in degrees Kelvin
#         e_sat   is the daily 2m saturation vapour pressure in mbar
#         w_sat   is the daily 2m saturation mixing ratio in g/kg
#         w       is the daily 2m mixing ratio in g/kg
#         theta_E is the daily temperature adiabatically relocated to 1000mbar in degrees Kelvin
#         T_wbt   is the daily 2m natural wet bulb temperature in degrees Celcius
        
#     Output variables:
#         T_wbgt  is the daily 2m wet bulb globe temperature in degrees Celcius
#     """
    
#     T_wbgt = T_a = T_L = e_sat = w_sat = w = theta_E = T_wbt = rh_ref_clip = None
        
#     if p_ref is not None:
#         rh_ref_clip = np.clip(rh_ref,0.01,100.0)

#         T_a     = T_ref - 273.15

#         #T_L     = 1.0 / ( T_ref - 55 - np.log(rh_ref_clip/100.0)/2840.0 ) + 55.0 # bug1
#         T_L     = 1.0 / ( 1.0/(T_ref - 55) - np.log(rh_ref_clip/100.0)/2840.0 ) + 55.0 # Bolton (1980, MWR) equation 22

#         e_sat   = np.exp(                          -2991.2729/(T_ref**2.0)                          - 6017.0128/T_ref                          + 18.87643854                          - 0.028354721*T_ref                          + (T_ref**2.0)*1.7838310e-5                          - (T_ref**3.0)*8.4150417e-10                          + (T_ref**4.0)*4.4412543e-13                          + 2.858487*np.log(T_ref)                         ) / 100.0
#                         # + (T_ref**2.0)*1.7838310e-7 - bug2 see table 1 in Wexler (1976) 

#         w_sat   = 621.97*e_sat/(p_ref-e_sat) # ?? or is it w_sat = 621.97*e_sat/(p_ref-621.97) derived from equation 16 in Bolton (1980 MWR)

#         w       = rh_ref_clip / 100.0 * w_sat

#         theta_E = T_ref * ( np.power(1000.0/p_ref, 0.2854*(1.0-w*0.28e-3) ) ) * np.exp( (3.376/T_L-0.00254) * w * (1.0 + w*0.81e-3) )

#         #T_wbt   = 45.114 - 51489*np.power(theta_E/273.15, -3.504) # bug3 ??
#         T_wbt   = 45.114 - 51.489*np.power(theta_E/273.15, -3.504)   # equation 3.3 in Davies-Jones (2008, MWR)

#         T_wbgt  = 0.7*T_wbt + 0.3*T_a
        
#     else:
#         print('### Using simple expression since mean sea level pressure is not provided ###')
#         rh_ref_clip = np.clip(rh_ref,0.01,100.0)
#         T_a     = T_ref - 273.15
#         T_wbgt =             T_a * np.arctan(0.151977*np.sqrt(rh_ref_clip + 8.313659))  + np.arctan(rh_ref_clip + T_a)             + np.arctan(rh_ref_clip - 1.676331)             + 0.00391838*(rh_ref_clip**1.5)*np.arctan(0.023101*rh_ref_clip)             - 4.686035
        
#     return T_wbgt, T_a, T_L, e_sat, w_sat, w, theta_E, T_wbt, rh_ref_clip

## Read BARRA2 data 

In [15]:
## barra2 
start_date          = '2007-01-01' #start date of extracting historic climate variables
start_date_analysis = '2007-01-01' #start date of health data
end_date            = '2023-12-31' ### --> infl

In [16]:
population_gridded_filename = path + 'data/climate_data/gpw_v4_population_density_rev11_1_deg.nc'
ds_pop_raw = xr.open_dataset(population_gridded_filename)
ds_pop_raw = ds_pop_raw.rename({'Population Density, v4.11 (2000, 2005, 2010, 2015, 2020): 1 degree':'popn'})

ds_pop_raw = ds_pop_raw.rename({'longitude':'lon'}).rename({'latitude':'lat'}).sortby('lat').sortby('lon') #20 raster points
ds_pop_raw = ds_pop_raw.sel(raster=slice(0,5))   #why (0,5)
ds_pop_raw = ds_pop_raw.rename({'raster':'time'})
time = ((ds_pop_raw['time'].values-1)*5+2000).astype(str).astype('datetime64[Y]').astype('datetime64[M]').astype('datetime64[D]') 
#time = ['2000-01-01', '2005-01-01', '2010-01-01', '2015-01-01','2020-01-01']
ds_pop_raw = ds_pop_raw.assign_coords(time=time) 

ds_pop_raw #from 2000-01-01 to 2020-01-01 (lon: 360 lat: 180 time: 5)

<xarray.Dataset> Size: 1MB
Dimensions:  (time: 5, lat: 180, lon: 360)
Coordinates:
  * time     (time) datetime64[s] 40B 2000-01-01 2005-01-01 ... 2020-01-01
  * lat      (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon      (lon) float64 3kB -179.5 -178.5 -177.5 -176.5 ... 177.5 178.5 179.5
Data variables:
    popn     (time, lat, lon) float32 1MB ...
Attributes:
    proj4:        +proj=longlat +datum=WGS84 +no_defs +ellps=WGS84 +towgs84=0...
    Conventions:  CF-1.4
    created_by:   R, packages ncdf4 and raster (version 2.8-4)
    date:         2018-11-16 09:56:26

In [17]:
# ########## files to read Baraa2
tas_file = path + 'data/climate_data/barra2/barra2.AUS11_ERAS5.tas.hres.month_average.nc' #historic avg temperature data file
# tasmax_file = path + 'data/climate_data/barra2/barra2.AUS11_ERAS5.tasmax.hres.month_average.nc' #historic max temperature data file
# tasmin_file = path + 'data/climate_data/barra2/barra2.AUS11_ERAS5.tasmin.hres.month_average.nc' #historic max temperature data file
rh_file = path + 'data/climate_data/barra2/barra2.AUS11_ERAS5.hurs.hres.month_average.nc' #historic relative humidity data file
# pr_file = path + 'data/climate_data/barra2/barra2.AUS11_ERAS5.pr.hres.month_average.nc' #historic precipitation data file


############ read Barra2 data ----------------------
print('Reading BARRA monthly')

## read temperature: date range in original file 2007-01-01 to 2023-12-01 for barra2
print('   tas')
ds_tas = xr.open_mfdataset(tas_file,  engine="h5netcdf")
ds_tas = ds_tas.drop(['time_bnds','height']) #.rename({'tas':'T_ref'})
time = ds_tas['time'].values.astype('datetime64[M]').astype('datetime64[D]') #204 time points
ds_tas = ds_tas.assign_coords(time=time)
# ds_tas

# print('   tasmax')
# ds_tasmax = xr.open_mfdataset(tasmax_file)
# ds_tasmax = ds_tasmax.drop(['time_bnds','height', 'bnds'])  
# time = ds_tasmax['time'].values.astype('datetime64[M]').astype('datetime64[D]') #204 time points
# ds_tasmax = ds_tasmax.assign_coords(time=time)
# # ds_tasmax

# print('   tasmin')
# ds_tasmin = xr.open_mfdataset(tasmin_file)
# ds_tasmin = ds_tasmin.drop(['time_bnds','height', 'bnds'])  
# time = ds_tasmin['time'].values.astype('datetime64[M]').astype('datetime64[D]') #204 time points
# ds_tasmin = ds_tasmin.assign_coords(time=time)
# # ds_tasmin


print('   rh')
ds_rh = xr.open_mfdataset(rh_file, engine="h5netcdf") 
ds_rh = ds_rh.rename({'hurs':'RH'}).drop(['height', 'level_height', 'model_level_number', 'sigma', 'time_bnds'])
time = ds_rh['time'].values.astype('datetime64[M]').astype('datetime64[D]')
ds_rh = ds_rh.assign_coords(time=time)
# ds_rh

# print('   pr')
# ds_pr = xr.open_mfdataset(pr_file) 
# ds_pr = ds_pr.drop(['bnds', 'time_bnds'])
# time = ds_pr['time'].values.astype('datetime64[M]').astype('datetime64[D]')
# ds_pr = ds_pr.assign_coords(time=time)
# ds_pr


# #### merge all climate variables   
print('   merging')
ds_climate_raw = xr.merge([ds_tas, ds_rh]).sortby('lat').sortby('lon') ## xr.merge([ds_tas, ds_tasmax, ds_tasmin,ds_rh, ds_pr]).sortby('lat').sortby('lon')
# del ds_tas, ds_tasmax, ds_tasmin, ds_rh,ds_pr
del ds_tas, ds_rh
ds_climate_raw 




Reading BARRA monthly
   tas
   rh
   merging


<xarray.Dataset> Size: 2GB
Dimensions:  (time: 204, lat: 646, lon: 1082)
Coordinates:
  * time     (time) datetime64[s] 2kB 2007-01-01 2007-02-01 ... 2023-12-01
  * lat      (lat) float64 5kB -57.97 -57.86 -57.75 -57.64 ... 12.76 12.87 12.98
  * lon      (lon) float64 9kB 88.48 88.59 88.7 88.81 ... 207.2 207.3 207.4
Data variables:
    tas      (time, lat, lon) float64 1GB dask.array<chunksize=(34, 108, 181), meta=np.ndarray>
    RH       (time, lat, lon) float64 1GB dask.array<chunksize=(34, 108, 181), meta=np.ndarray>
Attributes: (12/59)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1M.json
    productive_version:        738bb56
    variable_version:          v20231001
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    geospatial_lat_max:        12.98
    geospatial_lat_units:      degrees_north
    geospatial_lon_min:        88.48
    geospatial_lon_max:        207.39
    geospatial_lon_units:      degrees_east
    history:                   Sat May 04 16:06:31 2024: /g/data/access/ngm/m...

In [18]:
## Interpolate exposure and hazard onto a common grid
ds_pop_raw = ds_pop_raw.sel(lon=slice(-68+180,-25+180)).sel(lat=slice(-45,-7))# population grid 
ds_climate_raw = ds_climate_raw.sel(lon=slice(-68+180,-25+180)).sel(lat=slice(-45,-7))
ds_climate_raw = ds_climate_raw.sel(time=slice(start_date,end_date))

ds_climate_raw ### jiraa time: 156 lon: 35 lat: 31

min_lat = max(np.min(ds_pop_raw.lat.values), np.min(ds_climate_raw.lat.values))
max_lat = min(np.max(ds_pop_raw.lat.values), np.max(ds_climate_raw.lat.values))
lat_new = np.linspace(min_lat, max_lat, len(ds_climate_raw.lat.values)) #len(ds_pop_raw.lat.values) =38 ####  -- use this for Baraa2
print([min_lat, max_lat]) #[-44.5, -7.5]

min_lon = max(np.min(ds_pop_raw.lon.values), np.min(ds_climate_raw.lon.values))
max_lon = min(np.max(ds_pop_raw.lon.values), np.max(ds_climate_raw.lon.values))
lon_new = np.linspace(min_lon, max_lon, len(ds_climate_raw.lon.values)) #len(ds_pop_raw.lon.values) = 43  -- use this for Baraa2
print([min_lon, max_lon]) #[112.5, 154.5]

print('interp pop')
ds_pop     = ds_pop_raw.interp(lon=lon_new, method='linear').interp(lat=lat_new, method='linear').compute() #5 time points
print('interp clim')
ds_climate = ds_climate_raw.interp(lon=lon_new, method='linear').interp(lat=lat_new, method='linear').compute() #240 time points


ds_pop ##  # time: 5 lat: 346 lon: 39 ; 15 (yrly scale)  time points (2000-01-01 to   2020-01-01)
ds_climate ## 204 (monthly scale) time points (2007-01-01 to 2023-12-01)

[np.float64(-44.5), np.float64(-7.5)]
[np.float64(112.5), np.float64(154.5)]
interp pop
interp clim


<xarray.Dataset> Size: 442MB
Dimensions:  (time: 204, lat: 346, lon: 391)
Coordinates:
  * time     (time) datetime64[s] 2kB 2007-01-01 2007-02-01 ... 2023-12-01
  * lat      (lat) float64 3kB -44.5 -44.39 -44.29 -44.18 ... -7.714 -7.607 -7.5
  * lon      (lon) float64 3kB 112.5 112.6 112.7 112.8 ... 154.3 154.4 154.5
Data variables:
    tas      (time, lat, lon) float64 221MB 283.1 283.1 283.1 ... 301.9 301.9
    RH       (time, lat, lon) float64 221MB 78.26 78.26 78.25 ... 77.94 77.94
Attributes: (12/59)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1M.json
    productive_version:        738bb56
    variable_version:          v20231001
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    geospatial_lat_max:        12.98
    geospatial_lat_units:      degrees_north
    geospatial_lon_min:        88.48
    geospatial_lon_max:        207.39
    geospatial_lon_units:      degrees_east
    history:                   Sat May 04 16:06:31 2024: /g/data/access/ngm/m...

In [20]:
# T_wbgt = calculate_WBGT(None, ds_climate.tas.values, ds_climate.RH.values)[0] # Note, convert psl from Pa to mbar
ds_climate['tas'] = ds_climate['tas'] - 273 # convert from Kelvin to Celcius
# ds_climate['tasmax'] = ds_climate['tasmax'] - 273 # convert from Kelvin to Celcius
# ds_climate['tasmin'] = ds_climate['tasmin'] - 273 # convert from Kelvin to Celcius
# ds_climate['wet_bulb_temp']  = ds_climate['tas']*0.0 + T_wbgt
# ds_climate['time'].values[:] = np.array(np.array(ds_climate.time.values).astype('datetime64[M]') ).astype('datetime64[D]')
ds_climate = ds_climate.sel(time=slice(start_date,end_date))
# T_wbgt
# del T_wbgt
ds_climate


<xarray.Dataset> Size: 442MB
Dimensions:  (time: 204, lat: 346, lon: 391)
Coordinates:
  * time     (time) datetime64[s] 2kB 2007-01-01 2007-02-01 ... 2023-12-01
  * lat      (lat) float64 3kB -44.5 -44.39 -44.29 -44.18 ... -7.714 -7.607 -7.5
  * lon      (lon) float64 3kB 112.5 112.6 112.7 112.8 ... 154.3 154.4 154.5
Data variables:
    tas      (time, lat, lon) float64 221MB -262.9 -262.9 ... -244.1 -244.1
    RH       (time, lat, lon) float64 221MB 78.26 78.26 78.25 ... 77.94 77.94
Attributes: (12/59)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1M.json
    productive_version:        738bb56
    variable_version:          v20231001
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    geospatial_lat_max:        12.98
    geospatial_lat_units:      degrees_north
    geospatial_lon_min:        88.48
    geospatial_lon_max:        207.39
    geospatial_lon_units:      degrees_east
    history:                   Sat May 04 16:06:31 2024: /g/data/access/ngm/m...

In [21]:
## read the AU shape file and remove unwanted locations including ACT Other Territories','Outside Australia'
shape_file = gpd.read_file(path + 'data/shapeFiles/STE_2021_AUST_GDA2020.shp') #10 by 9
shape_file = shape_file[~shape_file['STE_NAME21'].isin(['Other Territories','Outside Australia'])] #remove unwanted regions
print(shape_file.shape) #8 by 9
shape_file

(8, 9)


,STE_CODE21,STE_NAME21,CHG_FLAG21,CHG_LBL21,AUS_CODE21,AUS_NAME21,AREASQKM21,LOCI_URI21,geometry
0,1,New South Wales,0,No change,AUS,Australia,8.007977e+05,http://linked.data.gov.au/dataset/asgsed3/STE/1,"MULTIPOLYGON (((159.0623 -31.50886, 159.06218 ..."
1,2,Victoria,0,No change,AUS,Australia,2.274962e+05,http://linked.data.gov.au/dataset/asgsed3/STE/2,"MULTIPOLYGON (((146.29286 -39.15778, 146.29341..."
2,3,Queensland,0,No change,AUS,Australia,1.730171e+06,http://linked.data.gov.au/dataset/asgsed3/STE/3,"MULTIPOLYGON (((142.5314 -10.68301, 142.53072 ..."
3,4,South Australia,0,No change,AUS,Australia,9.842314e+05,http://linked.data.gov.au/dataset/asgsed3/STE/4,"MULTIPOLYGON (((140.66025 -38.06256, 140.66006..."
4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5,"MULTIPOLYGON (((117.86953 -35.19108, 117.86961..."
5,6,Tasmania,0,No change,AUS,Australia,6.801754e+04,http://linked.data.gov.au/dataset/asgsed3/STE/6,"MULTIPOLYGON (((144.60439 -41.01001, 144.60443..."
6,7,Northern Territory,0,No change,AUS,Australia,1.348134e+06,http://linked.data.gov.au/dataset/asgsed3/STE/7,"MULTIPOLYGON (((133.02818 -10.90839, 133.02982..."
7,8,Australian Capital Territory,0,No change,AUS,Australia,2.358133e+03,http://linked.data.gov.au/dataset/asgsed3/STE/8,"POLYGON ((149.06239 -35.1591, 149.09134 -35.14..."


In [22]:
#### create a mask for AU states/territories

num_regions = len(list(shape_file.STE_NAME21)) #7 regions -- for influ
print(num_regions)
numbers=list(range(0,num_regions)) #[0, 1, 2, 3, 4, 5, 6]  
print(numbers)
outlines=list(shape_file.geometry.values[i] for i in range(0,num_regions)) #gives multipolygon values 
mask_poly = regionmask.Regions(name='mask', numbers=numbers, names=list(shape_file.STE_NAME21), abbrevs=list(shape_file.STE_CODE21), outlines=outlines) ###--flu
 
#create a 2D float mask of a set of regions for the given lat/ lon grid - do this for both ds_pop and ds_climate arrays
mask_reanalysis_pop = mask_poly.mask(ds_pop.isel(time = 0), lat_name='lat', lon_name='lon')
mask_reanalysis = mask_poly.mask(ds_climate.isel(time = 0), lat_name='lat', lon_name='lon') #gives a mask_2Dfloat xarray.DataArray


8
[0, 1, 2, 3, 4, 5, 6, 7]


TypeError: Regions.mask() got an unexpected keyword argument 'lat_name'

In [ ]:
##### select population and climate variables based on state_shape_file regions
ds_pop_interp_region = ds_pop.popn
# ds_pop.sel(time='2020-01-01')

# region_name='Australia'
region_name='New South Wales'
#region_name='South Australia'
#region_name='Australian Capital Territory'

##### for influenza
if region_name=='Australia':
    ds_climate_region = ds_climate.where(mask_reanalysis <num_regions+1) #total 7 regions were there (240 time points)
    ds_pop_interp_region = ds_pop_interp_region.where(mask_reanalysis<num_regions+1)
    # ds_pop_interp_region = ds_pop_interp_region.where(mask_reanalysis_pop<8) #this give same plot as above
else:
    print(shape_file[shape_file.STE_NAME21==region_name].index[0])
    ds_climate_region = ds_climate.where(mask_reanalysis == shape_file[shape_file.STE_NAME21==region_name].index[0]) ##time: 156 lat: 346 lon: 391
    ds_pop_interp_region = ds_pop_interp_region.where(mask_reanalysis == shape_file[shape_file.STE_NAME21==region_name].index[0]) ##time: 5lat: 346 lon: 391

# region_name='New South Wales'
# shape_file[shape_file.STE_NAME21==region_name].index[0]## 0

In [ ]:
ds_climate_region

In [ ]:
plt.figure()

ax = plt.subplot(5,2,1)
mask_reanalysis.plot(ax = ax) #regions
# plt.ylim(-30,-10) ## mrsa 

ax = plt.subplot(5,2,2)
ds_climate_region.tas.sel(time=slice('2008-01-01','2019-12-01')).mean('time').plot(ax = ax, cbar_kwargs={'label':'TAS'}, cmap = 'coolwarm')
plt.title(' ')
plt.title('(a)', loc='left');
plt.xlabel('longitude ($^\circ$E)')
plt.ylabel('latitude ($^\circ$N)')
# plt.legend(labels = 'TASmax')
# plt.ylim(-30,-10) ## mrsa 

# ds_pop_interp_region.attrs.pop('long_name')  #this removes long_name from attributes
ax = plt.subplot(5,2,3)
# ds_pop_interp_region.isel(time=-1).plot(ax = ax) # population density
np.log(ds_pop_interp_region.isel(time=-1)).plot(ax = ax, cbar_kwargs={'label':'People per square kilometer'},  cmap = 'coolwarm')
plt.xlabel('longitude ($^\circ$E)')
plt.ylabel('latitude ($^\circ$N)')
plt.title(' ')
plt.title('(b)', loc='left');
# plt.ylim(-30,-10) ## mrsa 

# # ##### average over regions
ax = plt.subplot(5,2,4)
# area_avg = ds_climate_region.tasmax.mean('lat').mean('lon')
area_avg = ds_climate_region.tas.mean('lat').mean('lon')
# static_pop_avg = (ds_pop_interp_region.mean('time')*ds_climate_region.tasmax/ds_pop_interp_region.mean('time').sum('lat').sum('lon')).sum('lat').sum('lon')
static_pop_avg = (ds_pop_interp_region.mean('time')*ds_climate_region.tas/ds_pop_interp_region.mean('time').sum('lat').sum('lon')).sum('lat').sum('lon')                                                 
plt.plot(ds_climate_region.time, area_avg, 'k-', label='Area weighted avg')
plt.plot(ds_climate_region.time, static_pop_avg, 'r-', label='static pop weighted avg')
plt.title('Average maximum temperature (TAS)')
plt.legend()

plt.tight_layout()

In [ ]:
ds_climate_avg, ds_climate_pop_weighted_avg = aggregate_climate_data_per_jurisdiction(ds_climate, ds_pop.sel(time='2020-01-01'), shape_file, mask_reanalysis)

In [ ]:
##save data as .nc file 
 
# # print(path + 'data/climate_data/barra2/barra2_allAU.nc')
# ds_climate_pop_weighted_avg.to_netcdf(path + 'data/climate_data/barra2/barra2_allAU.nc')

In [ ]:
# ds_climate_pop_weighted_avg
states = ['NSW', 'NT', 'QLD', 'SA', 'TAS', 'VIC', 'WA']
i=0
print(states[i])
ds_cv = ds_climate_pop_weighted_avg.tasmax.sel(time=slice('2008-01-01','2019-12-01')).sel(jurisdiction= states[i])
ds_cv.values
plt.figure()

ax = plt.subplot(5,2,1)
plt.plot(ds_cv.time, ds_cv.values, 'b-', label= [states[i]])
plt.grid(linestyle=':')
plt.ylabel("Monthly TASmax", fontsize=16) 
plt.xlabel("Time", fontsize=12) 
plt.xticks(rotation= 60) #'vertical'
plt.margins(0.02)
plt.legend()
plt.tight_layout()

### convert to dataframe and add seasons, month, year 

In [ ]:
ds_climate_pop_weighted_avg## 156 time points Jan 2007 to dec 2019 (13yrs*12 = 156)

In [ ]:
2019-2007
13*12*8

In [ ]:

# df_barra2 = ds_climate_pop_weighted_avg.to_dataframe().reset_index()
# print(df_barra2.shape)   ## 13 yrs*12*8locations (includeing AU) = 1248
# df_barra2.insert(2,'year', df_barra2['time'].dt.strftime('%Y'))  ##year  is string format
# df_barra2['year'] = df_barra2['year'].astype('int64')
# df_barra2.insert(3,'month', df_barra2['time'].dt.strftime('%m')) ## month is string format
# df_barra2['month'] = df_barra2['month'].astype('int64')

# inset season column
season_mapping = {
             '9' : 'spring','10': 'spring', '11': 'spring',
             '12': 'summer', '1': 'summer', '2': 'summer',
              '3': 'autumn', '4': 'autumn', '5':   'autumn',
              '6': 'winter', '7': 'winter', '8':  'winter'
}

df_barra2['season'] = df_barra2['month'].apply(str).map(season_mapping)

print(df_barra2.dtypes)
# print(df_barra2.columns)
# print(df_barra2.head(1))
# print(df_barra2.tail(1))
df_barra2

In [ ]:
df_barra2 = df_barra2.sort_values(['jurisdiction', 'time'], ascending=[True, True])
df_barra2

In [ ]:
#save to csv
df_barra2.to_csv(path + 'data/climate_data/barra2/barra2_allAU.csv', index = False)

In [ ]:
check = pd.read_csv(path + 'data/climate_data/barra2/barra2_allAU.csv', low_memory=False) 
print(check.dtypes)
check